## Objectif de cette analyse

L’objectif de cette analyse est de comprendre la répartition géographique des niveaux de vie et d’identifier les disparités socio-économiques entre territoires. À partir du revenu médian, du taux de pauvreté et des indices d’inégalités on voudrait être capable de :
- Mesurer les contrastes régionaux entre zones aisées et zones défavorisées

- Analyser la structure du revenu (part des salaires, retraites, aides sociales) selon les départements

- Observer le lien entre pauvreté et inégalités

In [4]:
# install necessary packages
install.packages("readxl")
install.packages("dplyr")
install.packages("tidyr")
install.packages("ggplot2")
install.packages("maps")
install.packages("mapproj")
install.packages("FactoMineR")
install.packages("factoextra")
install.packages("gplots")
install.packages("corrplot")
install.packages("ellipse")
install.packages("car")
install.packages("lmtest")
install.packages("writexl")

# Load necessary packages
library(readxl)
library(dplyr)
library(tidyr)
library(ggplot2)
library(maps)
library(mapproj)
library(FactoMineR)
library(factoextra)
library(gplots)
library(corrplot)
library(ellipse)
library(car)
library(lmtest)
library(writexl)



# Plotting options
options(repr.plot.width = 14, repr.plot.height = 8)

also installing the dependency ‘cpp11’





The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages 

## Présentation du jeu de données

Le jeu de données que nous analisions provient de la base FILOSOFI 2018 (Fichier Localisé Social et Fiscal) publiée par l’INSEE. 
Cette base combine des données fiscales anonymisées issues de la Direction Générale des Finances Publiques (DGFiP) avec des données démographiques de l’INSEE.

In [6]:
# Read the data from the Excel file
data <- read_excel("datasets.xlsx", sheet = 1)

# Convert the data to a data frame
data <- as.data.frame(data)

dim(data)
names(data)

[1] 12395    30

[1] "IRIS"                                                 
 [2] "Libellé de l'IRIS"                                    
 [3] "Commune ou ARM"                                       
 [4] "Libellé commune ou ARM"                               
 [5] "Département"                                          
 [6] "Libellé département"                                  
 [7] "Part des ménages fiscaux imposés (%)"                 
 [8] "Taux de bas revenus déclarés au seuil de 60 % (%)"    
 [9] "1er quartile (€)"                                     
[10] "Médiane (€)"                                          
[11] "3e quartile (€)"                                      
[12] "Écart inter-quartile rapporté à la médiane"           
[13] "1er décile (€)"                                       
[14] "2e décile (€)"                                        
[15] "3e décile (€)"                                        
[16] "4e décile (€)"                                        
[17] "6e décile (€)"                                        
[18] "7e décile (€)"                                        
[19] "8e décile (€)"                                        
[20] "9e décile (€)"                                        
[21] "Rapport interdécile D9/D1"                            
[22] "S80/S20"                                              
[23] "Indice de Gini"                                       
[24] "Part des revenus d’activité (%)"                      
[25] "dont part des salaires et traitements (%)"            
[26] "dont part des indemnités de chômage (%)"              
[27] "dont part des revenus des activités non salariées (%)"
[28] "Part des pensions, retraites et rentes (%)"           
[29] "Part des autres revenus (%)"                          
[30] "Note de précaution (voir onglet seuils)"

On renomme les colonnes avec des noms courts. Ce sont les codes officiels de l'INSEE :

| Code | Signification |
|---|---|
| PIMP | Part des ménages fiscaux imposés (%) |
| TP60 | Taux de bas revenus au seuil de 60 % (%), notre indicateur de pauvreté |
| Q1, MED, Q3 | 1er quartile, médiane, 3e quartile du revenu par UC (€) |
| D1 … D9 | Déciles du revenu par UC (€) |
| EQ | Écart interquartile / médiane |
| RD | Rapport interdécile D9/D1 |
| S80S20 | Ratio S80/S20 |
| GINI | Indice de Gini |
| PACT | Part des revenus d'activité (%) = PTSA + PCHO + PBEN |
| PTSA | dont salaires et traitements (%) |
| PCHO | dont indemnités de chômage (%) |
| PBEN | dont revenus des activités non salariées (%) |
| PPEN | Part des pensions, retraites et rentes (%) |
| PAUT | Part des autres revenus, surtout du patrimoine (%) |
| NOTE | Note de précaution de l'INSEE |

In [9]:
# Rename the columns
rename_columns <- data
names(rename_columns) <- c("IRIS", "LIBIRIS", "COM", "LIBCOM", "DEP", "LIBDEP", "PIMP", "TP60", "Q1", "MED", "Q3", "EQ", "D1", "D2", "D3", "D4", "D6", "D7", "D8", "D9", "RD", "S80S20", "GINI", "PACT", "PTSA", "PCHO", "PBEN", "PPEN", "PAUT", "NOTE")

# Check the structure and first few rows of the renamed data frame
str(rename_columns)
head(rename_columns)

'data.frame':	12395 obs. of  30 variables:
 $ IRIS   : chr  "010040101" "010040102" "010040201" "010040202" ...
 $ LIBIRIS: chr  "Les Perouses-Triangle d'Activité" "Longeray-Gare" "Centre-Saint-Germain-Vareilles" "Tiret-Les Allymes" ...
 $ COM    : chr  "01004" "01004" "01004" "01004" ...
 $ LIBCOM : chr  "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" ...
 $ DEP    : chr  "01" "01" "01" "01" ...
 $ LIBDEP : chr  "Ain" "Ain" "Ain" "Ain" ...
 $ PIMP   : num  45.4 39.3 43.7 61.1 41.9 46.7 48.6 39.2 36.7 54.5 ...
 $ TP60   : num  31 39.4 28.8 14.6 25.9 21.2 20.6 34.5 29.9 11.4 ...
 $ Q1     : num  11150 8890 11830 17180 12660 ...
 $ MED    : num  18540 16180 18870 24780 19590 ...
 $ Q3     : num  25900 23290 26560 33280 28730 ...
 $ EQ     : num  0.8 0.89 0.78 0.65 0.82 0.86 0.84 0.91 0.89 0.75 ...
 $ D1     : num  5500 3800 6770 10380 6620 ...
 $ D2     : num  9490 7690 9990 15570 10530 ...
 $ D3     : num  12830 10230 13210 19020 13900 ...
 $ D4     : num

,IRIS,LIBIRIS,COM,LIBCOM,DEP,LIBDEP,PIMP,TP60,Q1,MED,⋯,RD,S80S20,GINI,PACT,PTSA,PCHO,PBEN,PPEN,PAUT,NOTE
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,010040101,Les Perouses-Triangle d'Activité,01004,Ambérieu-en-Bugey,01,Ain,45.4,31.0,11150,18540,⋯,6.2,7.2,0.332,68.9,63.2,3.1,2.6,27.3,3.8,0
2,010040102,Longeray-Gare,01004,Ambérieu-en-Bugey,01,Ain,39.3,39.4,8890,16180,⋯,8.3,9.8,0.366,69.6,62.9,4.5,2.2,26.9,3.5,0
3,010040201,Centre-Saint-Germain-Vareilles,01004,Ambérieu-en-Bugey,01,Ain,43.7,28.8,11830,18870,⋯,5.2,7.4,0.347,67.7,60.6,3.1,4.0,28.3,4.0,0
4,010040202,Tiret-Les Allymes,01004,Ambérieu-en-Bugey,01,Ain,61.1,14.6,17180,24780,⋯,4.3,7.3,0.387,62.7,57.5,2.1,3.1,21.2,16.1,0
5,010330102,Centre Ville,01033,Valserhône,01,Ain,41.9,25.9,12660,19590,⋯,6.2,8.6,0.365,70.2,63.8,4.1,2.3,25.8,4.0,0
6,010330103,Lancrans-Coupy-Vanchy,01033,Valserhône,01,Ain,46.7,21.2,14700,22970,⋯,6.3,7.4,0.358,75.8,70.4,3.5,1.9,19.4,4.8,0
